# 05 · `gl_engine/resolve/resolver.py`

## What this file is for

ISO doesn't file one rulebook. It files a national one and then a separate package per state, and it refiles them constantly — this corpus holds several editions of every jurisdiction, going back years and forward past today.

So before anything can be rated, one question has to be answered: **for this state, on this date, which two packages govern?** That is the whole job of this file. It picks the state package, then finds the national package that state package *declares* as its parent.

It is 145 lines, and its own docstring says every one of its five steps has a tempting shortcut that is wrong. This notebook walks each of them and shows what the shortcut would have cost.

**Depends on:** [`01-config`](01-config.ipynb), [`02-errors`](02-errors.ipynb), [`04-erc-discovery`](04-erc-discovery.ipynb).

## Its public surface

Generated from the module rather than typed here, so it can't drift from the code.

In [ ]:
import inspect, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from gl_engine.resolve import resolver

for name, obj in vars(resolver).items():
    if name.startswith("_") or getattr(obj, "__module__", None) != resolver.__name__:
        continue
    if inspect.isclass(obj):
        print(f"class {name}")
        for m, f in vars(obj).items():
            if not m.startswith("_") and callable(f):
                print(f"    .{m}{inspect.signature(f)}")
            elif not m.startswith("_") and isinstance(f, property):
                print(f"    .{m}  (property)")
    elif inspect.isfunction(obj):
        print(f"def {name}{inspect.signature(obj)}")

## The smallest thing that works

Ask what governs Georgia on 11 August 2026.

In [ ]:
from gl_engine import EditionResolver

r = EditionResolver()              # reads the corpus once; reuse it
res = r.resolve("GA", "20260811")

print(res)
print()
print("state package :", res.state.pkg_id)
print("its parent    :", res.parent.pkg_id)
print("layer order   :", [p.pkg_id for p in res.layers])

`layers` is ordered **state first**. That is the override order: when both packages define a rule or a table of the same name, the state's wins outright — and a state may override with something deliberately empty, which means *not offered here* rather than *look upstream*.

Everything downstream — [`06-resolve-book`](06-resolve-book.ipynb), the interpreter, the kernel — consumes exactly this pair.

## The interesting case

### "Latest" is never "now"

The corpus holds editions effective *after* today. Taking the newest one would rate today's risk against rules that aren't in force yet.

In [ ]:
asof = "20260811"

future = {j: [p.pkg_id for p in r.by_juris[j] if p.identity.edition > asof]
          for j in r.jurisdictions}
future = {j: pkgs for j, pkgs in future.items() if pkgs}

print(f"jurisdictions holding editions dated after {asof} : {len(future)}")
print(f"future-dated packages in total                    : "
      f"{sum(len(p) for p in future.values())}")
print()

# take the jurisdiction with the most, and watch the shortcut fail
worst        = max(future, key=lambda j: len(future[j]))
newest_filed = r.by_juris[worst][-1].pkg_id            # the tempting shortcut
in_force     = r.resolve(worst, asof).state.pkg_id     # what actually governs

print(f"{worst}: newest edition ISO has filed   -> {newest_filed}")
print(f"{worst}: the one in force on {asof}  -> {in_force}")
print(f"\nsame package? {newest_filed == in_force}")
print("Every figure taken over the newest edition describes the future.")

### The parent is *declared*, not assumed

The step most likely to be got wrong. Each state package names its own national parent in its XSD, and **that is not always the newest national edition**. More than one national edition is live at any moment.

In [ ]:
asof = "20260811"
parents = r.declared_parents(asof)

print(f"national editions live on {asof}: {len(parents)}\n")
for pkg, states in sorted(parents.items()):
    print(f"  {pkg}  <- {len(states):>2} jurisdictions   {', '.join(sorted(states)[:8])}"
          + ("..." if len(states) > 8 else ""))

newest = max(parents)
not_newest = {p: s for p, s in parents.items() if p != newest}
print(f"\nStates NOT on the newest national edition: "
      f"{sum(len(s) for s in not_newest.values())}")
print("Nothing but reading the declaration catches these.")

This is also why **pinning a carrier to an older edition is safe**: pin the state package, and its declared parent comes with it automatically. Pair an old state package with today's national one and you'd get a complete, plausible, wrong premium.

## What it refuses

Three refusals, and each would otherwise produce a confident wrong answer. Run them — the exception *is* the documentation.

In [ ]:
from gl_engine.errors import ResolutionError

for label, juris, date in [
    ("a date below the corpus floor", "GA", "20200101"),
    ("a jurisdiction that isn't one", "ZZ", "20260811"),
    ("a malformed date",              "GA", "2026-08-11"),
]:
    try:
        r.resolve(juris, date)
        print(f"{label}: NO ERROR -- worth investigating")
    except ResolutionError as e:
        print(f"{label}:\n    {e}\n")

The first is the most interesting. Below `MIN_ASOF` the corpus can't resolve all 51 jurisdictions — so rather than serving a partial answer that looks complete, it refuses. A fourth refusal exists that this corpus can't currently trigger: if a state declares a parent that isn't present, the resolver raises instead of falling back to the newest national edition.

## Try it yourself

1. Resolve your own state on today's date, then on a date three years ago. Do you get different packages?
2. Find every jurisdiction whose declared parent is *not* the newest national edition. How many are there?
3. Pick one state and list its editions with their effective dates. How often does ISO refile it?
4. Find the earliest date on which all 51 jurisdictions still resolve. How close is it to `MIN_ASOF`?

In [ ]:
# your turn


<details>
<summary><b>Answers</b></summary>

```python
# 2
parents = r.declared_parents("20260811")
newest = max(parents)
odd = {p: s for p, s in parents.items() if p != newest}
print(odd)

# 3
for p in r.by_juris["GA"]:
    print(p.identity.edition, p.identity.version, p.pkg_id)

# 4
from gl_engine.config import MIN_ASOF
print(MIN_ASOF)
r.resolve_all(MIN_ASOF)   # raises on the first jurisdiction that will not
```
</details>